# Train a model on the data

In [1]:

from collections import Counter
import jsonlines
import pandas as pd
import numpy as np
from numpy.typing import ArrayLike
import os

from ignite.engine import Engine, Events
from ignite.handlers import ProgressBar
from ignite.metrics import Recall, Precision, Accuracy, Loss

import torch
from torch import nn
import torch.nn.functional as F
import torch_geometric
from torch_geometric.data import HeteroData
from torch_geometric.nn import GraphConv, SAGEConv, to_hetero, HeteroConv
from torch_geometric import transforms as T
from torch_geometric import seed_everything

from jazz_graph.data.reporting import inspect_degrees
from jazz_graph.training.logging import (
    ExperimentLogger,
    save_embeddings_handler,
    save_checkpoint_handler,
    run_evaluator_handler,
    log_experiment_handler,
    console_logging,
    binary_output_transform
)
from jazz_graph.data.graph_builder import CreateTensors, prune_island_nodes, make_jazz_data
from jazz_graph.model.model import JazzModel, LinkPredictionModel, NodeClassifier


/opt/conda/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
models_dir = '/workspace/local_data/graph_parquet_proto'
assert os.path.exists(models_dir)
create = CreateTensors(models_dir)

In [3]:
data = make_jazz_data(create)

In [4]:
# TODO: report on the data a little more concreately.
# E.g., who are the hub nodes? How many nodes have > 50 edges.
# how many nodes have < 6 edges? All these, by type.
# Get really fancy and visualize a sub-graph.

def frequency_of_n_labels(data: HeteroData):
    """Return frequency of number of labels in the data, i.e., what percentage have 1 label, 0 labels, etc."""
    count_by_row = data['performance'].y.sum(dim=1)
    n_samples = data['performance'].y.shape[0]
    counter = Counter((int(x) for x in (count_by_row)))
    for i in range(len(counter)):
        count = counter[i]
        freq = count / n_samples
        print(f"Num samples with {i} labels: {freq:.3f}")

In [5]:
data.metadata()

(['performance', 'song', 'artist'],
 [('artist', 'performs', 'performance'),
  ('performance', 'performing', 'song'),
  ('artist', 'composed', 'song'),
  ('performance', 'rev_performs', 'artist'),
  ('song', 'rev_performing', 'performance'),
  ('song', 'rev_composed', 'artist')])

In [6]:
create.label_names()

['Swing',
 'Big Band',
 'Contemporary Jazz',
 'Bop',
 'Easy Listening',
 'Post Bop',
 'Free Jazz',
 'Dixieland',
 'Fusion',
 'Cool Jazz',
 'Hard Bop',
 'Free Improvisation',
 'Soul-Jazz',
 'Vocal',
 'Jazz-Funk',
 'Avant-garde Jazz',
 'Smooth Jazz',
 'Modal',
 'Latin Jazz',
 'Jazz-Rock']

In [7]:
print(data)
print(
    f"The graph contains {'' if data.has_isolated_nodes() else 'no '}isolated nodes and",
    f"is {'directed' if data.is_directed() else 'undirected'}."
)
frequency_of_n_labels(data)
for style, count in (zip(create.label_names(), data['performance'].y.sum(dim=0))):
    print(f"  {style}: {int(count) / create._labels.shape[0]:.1%}")
    # Easy Listening is probably a mislabel by modern standards.


HeteroData(
  performance={
    x=[6495, 1],
    y=[6495, 20],
    train_mask=[6495],
    dev_mask=[6495],
    test_mask=[6495],
  },
  song={ x=[2137, 1] },
  artist={ x=[2576, 1] },
  (artist, performs, performance)={ edge_index=[2, 31577] },
  (performance, performing, song)={ edge_index=[2, 3727] },
  (artist, composed, song)={ edge_index=[2, 2417] },
  (performance, rev_performs, artist)={ edge_index=[2, 31577] },
  (song, rev_performing, performance)={ edge_index=[2, 3727] },
  (song, rev_composed, artist)={ edge_index=[2, 2417] }
)
The graph contains no isolated nodes and is undirected.
Num samples with 0 labels: 0.212
Num samples with 1 labels: 0.561
Num samples with 2 labels: 0.187
Num samples with 3 labels: 0.036
Num samples with 4 labels: 0.004
  Swing: 6.3%
  Big Band: 5.2%
  Contemporary Jazz: 0.0%
  Bop: 10.3%
  Easy Listening: 4.8%
  Post Bop: 3.3%
  Free Jazz: 0.6%
  Dixieland: 2.0%
  Fusion: 0.0%
  Cool Jazz: 6.7%
  Hard Bop: 12.3%
  Free Improvisation: 0.0%
  Soul-Jaz

In [8]:
inspect_degrees(data)

,performs,performing,composed,rev_performs,rev_performing,rev_composed
count,2576.000000,6495.000000,2576.000000,6495.000000,2137.000000,2137.000000
mean,12.258152,0.573826,0.938276,4.861740,1.744034,1.131025
std,21.611876,0.496112,3.213998,4.589389,1.700268,0.420291
min,0.000000,0.000000,0.000000,0.000000,1.000000,1.000000
25%,0.000000,0.000000,0.000000,3.000000,1.000000,1.000000
50%,6.000000,1.000000,0.000000,4.000000,1.000000,1.000000
75%,12.000000,1.000000,1.000000,6.000000,2.000000,1.000000
max,307.000000,2.000000,62.000000,44.000000,21.000000,5.000000


## Model

## Train Style Classifier

In [ ]:
from torch_geometric.loader import NeighborLoader

def train_indicies(mask):
    num_nodes = mask.shape[0]
    all_node_indicies = torch.arange(num_nodes)
    return all_node_indicies[mask]

train_loader = NeighborLoader(
    data,
    [15, 15, 15],
    batch_size=128,
    input_nodes=('performance', train_indicies(data['performance'].train_mask)),
    shuffle=True
)
dev_loader = NeighborLoader(
    data,
    [15, 15, 15],
    batch_size=128,
    input_nodes=('performance', train_indicies(data['performance'].dev_mask)),
    shuffle=True
)

### Train function

In [ ]:
class GNNTrainingLogic:
    """Define training step and eval steps."""
    def __init__(self, model, optimizer, criterion):
        self.device = next(model.parameters()).device
        self.model = model
        self.optimizer = optimizer
        self.criterion = criterion

    def train_step(self, engine, batch: HeteroData) -> dict:
        """Complete one step of gradient descent."""
        self.model.train()
        self.optimizer.zero_grad()
        batch.to(self.device)
        batch_size = batch['performance'].batch_size
        y_pred = model(batch)[:batch_size]
        y_true = batch['performance'].y[:batch_size].to(torch.float)
        loss = self.criterion(y_pred, y_true.to(torch.float))  # TODO: need to cast here is just an artifact of wrong type in data.
        loss.backward()
        self.optimizer.step()
        return {'loss': loss.item(), 'y_pred': y_pred.detach(), 'y_true': y_true.detach()}

    def eval_step(self, engine, batch: HeteroData) -> dict:
        """Complete one pass over a batch of data with no-grad and return results."""
        self.model.eval()
        batch.to(self.device)

        batch_size = batch['performance'].batch_size
        with torch.no_grad():
            y_pred = model(batch)[:batch_size]
            y_true = batch['performance'].y[:batch_size].to(torch.float)
            loss = self.criterion(y_pred, y_true.to(torch.float))
        return {'y_pred': y_pred, 'y_true': y_true}

model_config = {
    'hidden_dim': 128,
    'embed_dim': 64,
    'dropout': 0.5
}

model = NodeClassifier(
    JazzModel(
        data['performance'].num_nodes,
        data['artist'].num_nodes,
        data['song'].num_nodes,
        hidden_dim=model_config['hidden_dim'],
        embed_dim=model_config['embed_dim'],
        dropout=model_config['dropout'],
        metadata=data.metadata()
    ),
    hidden_dim=model_config['hidden_dim'],
    num_classes=len(create.label_names())
)


experiment_config = {
    'model': model_config,
    'dataset': models_dir,
    'lr': .001,
    'batch_size': train_loader.batch_size,
    'epochs': 2
}


criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=experiment_config['lr'])
trainer_logic = GNNTrainingLogic(model, optimizer, criterion)
# assert trainer_logic.train_step(None, batch) is not None, "This is really just checking that the forward pass works."

experiment_logger = ExperimentLogger(root='../experiments', run_name=f'gnn_classifier_{os.path.basename(models_dir)}', config=experiment_config)

trainer = Engine(trainer_logic.train_step)
# train_evaluator = Engine(trainer_logic.eval_step)  # Wastful! Takes extra time!
dev_evaluator = Engine(trainer_logic.eval_step)

metrics = {
    'accuracy': Accuracy(output_transform=binary_output_transform),
    'recall': Recall(output_transform=binary_output_transform),
    'precision': Precision(output_transform=binary_output_transform),
    'loss': Loss(criterion, output_transform=lambda out: (out['y_pred'], out['y_true']))
}

for name, metric in metrics.items():
    metric.attach(trainer, name)
    metric.attach(dev_evaluator, name)

progress_bar = ProgressBar()
progress_bar.attach(trainer, metric_names=['loss'])

# trainer.add_event_handler(Events.EPOCH_COMPLETED, run_evaluator_handler, train_evaluator, train_loader, "Training")
trainer.add_event_handler(Events.EPOCH_COMPLETED, console_logging, 'Training', trainer)
trainer.add_event_handler(Events.EPOCH_COMPLETED, run_evaluator_handler, dev_evaluator, dev_loader)
trainer.add_event_handler(Events.EPOCH_COMPLETED, log_experiment_handler, experiment_logger, 'train', trainer)
dev_evaluator.add_event_handler(Events.EPOCH_COMPLETED, log_experiment_handler, experiment_logger, 'dev', trainer)
dev_evaluator.add_event_handler(Events.EPOCH_COMPLETED, console_logging, 'Valiation', trainer)
trainer.add_event_handler(Events.COMPLETED, save_embeddings_handler, experiment_logger, model)
trainer.add_event_handler(Events.COMPLETED, experiment_logger.save_checkpoint)
trainer.run(train_loader, max_epochs=experiment_config['epochs'])

## Edge Prediction

Quick shot at writing an edge prediction model. 
Conceptually, a recommender system based on this prediction "who worked with whom, on what?"
Thus, it's a lot like asking Theolious Monk "What would you recommend?"

In [9]:
seed_everything(42)
performs_edge_count = data[('artist', 'performs', 'performance')].num_edges

data_config = {
    'dataset': models_dir,
    'val_frac': .3,
    'test_frac': .3,
    'disjoint_train_ratio': 0.3,
    'neg_sample_ratio': 2.0,
    'sampling': {'num_neighbors': [5, 5, 3]}
}

split_graph = T.RandomLinkSplit(
    num_val=int(performs_edge_count * data_config['val_frac']),
    num_test=int(performs_edge_count * data_config['test_frac']),
    disjoint_train_ratio=data_config['disjoint_train_ratio'],
    # neg_sampling_ratio=data_config['neg_sample_ratio'],
    add_negative_train_samples=False,
    edge_types=[('artist', 'performs', 'performance')],
    rev_edge_types=[('performance', 'rev_performs', 'artist')],
    is_undirected=True
)
train_data, dev_data, test_data = split_graph(data)

In [10]:
for edge in data.metadata()[1]:
    edges = data[edge].edge_index
    print(edge)
    unique = len(set(map(tuple, edges.t().tolist())))
    print("unique:", unique)
    print("total:", edges.shape[1])
    print("ratio:", (edges.shape[1] - unique) / edges.shape[1])

('artist', 'performs', 'performance')
unique: 31577
total: 31577
ratio: 0.0
('performance', 'performing', 'song')
unique: 3727
total: 3727
ratio: 0.0
('artist', 'composed', 'song')
unique: 2417
total: 2417
ratio: 0.0
('performance', 'rev_performs', 'artist')
unique: 31577
total: 31577
ratio: 0.0
('song', 'rev_performing', 'performance')
unique: 3727
total: 3727
ratio: 0.0
('song', 'rev_composed', 'artist')
unique: 2417
total: 2417
ratio: 0.0


In [11]:

def shared_label_edges(edge_data: HeteroData, edge_label_data: HeteroData, edge: str | tuple):
    d1_edges = {tuple(e) for e in edge_data[edge].edge_label_index.t().tolist()}
    d2_edges = {tuple(e) for e in edge_label_data[edge].edge_label_index.t().tolist()}
    shared_edges = d1_edges.intersection(d2_edges)
    return shared_edges

def shared_rev_edges(data1, data2):
    train_edges = {
        (a,b) for a,b in data1['performs'].edge_index.t().tolist()
    }

    val_edges = {
        (a,b) for a,b in data2['performs'].edge_label_index.t().tolist()
    }

    reverse_overlap = {(b,a) for (a,b) in val_edges} & train_edges
    print(len(reverse_overlap))

shared_rev_edges(train_data, dev_data)
shared = shared_label_edges(train_data, dev_data, 'performs')
assert not shared, f"There are {len(shared)} edges in train-dev performs."


4


In [12]:

# Manually remove reverse edges that correspond to supervision edges
def remove_reverse_leakage(split_data, edge_type, rev_edge_type):
    """Remove reverse edges that leak supervision information."""
    # Get supervision edges
    supervise_edges = split_data[edge_type].edge_label_index

    # Get reverse edge index
    rev_edge_index = split_data[rev_edge_type].edge_index

    # Create set of (src, dst) tuples to remove (reversed)
    edges_to_remove = set()
    for i in range(supervise_edges.size(1)):
        src, dst = supervise_edges[0, i].item(), supervise_edges[1, i].item()
        edges_to_remove.add((dst, src))  # Reverse for the reverse edge type

    # Filter reverse edges
    mask = torch.ones(rev_edge_index.size(1), dtype=torch.bool)
    for i in range(rev_edge_index.size(1)):
        src, dst = rev_edge_index[0, i].item(), rev_edge_index[1, i].item()
        if (src, dst) in edges_to_remove:
            mask[i] = False

    # Update the edge index
    split_data[rev_edge_type].edge_index = rev_edge_index[:, mask]
    return split_data

# dev_data = remove_reverse_leakage(
#     dev_data,
#     ('artist', 'performs', 'performance'),
#     ('performance', 'rev_performs', 'artist')
# )
# test_data = remove_reverse_leakage(
#     test_data,
#     ('artist', 'performs', 'performance'),
#     ('performance', 'rev_performs', 'artist')
# )

In [13]:
from torch_geometric.loader import LinkNeighborLoader

def edge_training_data_factory(data: HeteroData, data_config: dict) -> LinkNeighborLoader:
    sampling = data_config['sampling']
    edge_loader = LinkNeighborLoader(
        data=data,
        num_neighbors=sampling['num_neighbors'],
        neg_sampling_ratio=data_config['neg_sample_ratio'],
        edge_label_index=(('artist', 'performs', 'performance'), data['performs'].edge_label_index),
        edge_label=None,
        batch_size=128,
        shuffle=True
    )
    return edge_loader

edge_loader_train = edge_training_data_factory(train_data, data_config)
edge_loader_dev = edge_training_data_factory(dev_data, data_config)

/opt/conda/lib/python3.11/site-packages/torch_geometric/loader/link_neighbor_loader.py:252: UserWarning: Using 'NeighborSampler' without a 'pyg-lib' installation is deprecated and will be removed soon. Please install 'pyg-lib' for accelerated neighborhood sampling
  neighbor_sampler = NeighborSampler(


In [14]:
iter_edges = iter(edge_loader_dev)

In [15]:
batch = next(iter_edges)
label_edges = {
    tuple(e) for e in batch['artist','performs','performance']
    .edge_label_index.t().tolist()
}
mp_edges = {
    tuple(e) for e in batch['artist','performs','performance']
    .edge_index.t().tolist()
}
leaked = label_edges & mp_edges
print("Leaked edges:", len(leaked))

reverse_mp = {(b,a) for (a,b) in mp_edges}
reverse_leak = label_edges & reverse_mp
print("Reverse leaks:", len(reverse_leak))

Leaked edges: 0
Reverse leaks: 0


In [16]:
data['performance'].num_nodes

6495

In [17]:
model_config = {
    'num_performances': data['performance'].num_nodes,
    'num_artists': data['artist'].num_nodes,
    'num_songs': data['song'].num_nodes,
    'hidden_dim': 128,
    'embed_dim': 64,
    'dropout': 0.
}

model = LinkPredictionModel(JazzModel(
    model_config['num_performances'],
    model_config['num_artists'],
    model_config['num_songs'],
    hidden_dim=model_config['hidden_dim'],
    embed_dim=model_config['embed_dim'],
    dropout=model_config['dropout'],
    metadata=data.metadata()
))

In [18]:
batch = next(iter(edge_loader_train))
batch['performance'].n_id

tensor([   9,   18,   28,  ..., 3579, 5568,  153])

In [19]:
batch['performs'].edge_label
batch['performs'].edge_label
batch['performs'].edge_label_index.shape
model(batch).shape


torch.Size([384])

In [20]:
class GNNTrainingLogic:
    """Define training step and eval steps."""
    def __init__(self, model, optimizer, criterion):
        self.device = next(model.parameters()).device
        self.model = model
        self.optimizer = optimizer
        self.criterion = criterion

    def _extract_model_args(self, batch):
        return batch.x_dict, batch.edge_index_dict, batch['performs'].edge_label_index

    def train_step(self, engine, batch: HeteroData) -> dict:
        """Complete one step of gradient descent."""
        self.model.train()
        self.optimizer.zero_grad()
        batch.to(self.device)

        y_pred = self.model(batch)
        y_true = batch['performs'].edge_label
        loss = self.criterion(y_pred, y_true)
        loss.backward()
        self.optimizer.step()
        return {'loss': loss.item(), 'y_pred': y_pred.detach(), 'y_true': y_true.detach()}

    def eval_step(self, engine, batch: HeteroData) -> dict:
        """Complete one pass over a batch of data with no-grad and return results."""
        self.model.eval()
        batch.to(self.device)
        with torch.no_grad():
            y_pred = self.model(batch)
            y_true = batch['performs'].edge_label
        return {'y_pred': y_pred, 'y_true': y_true}

experiment_config = {
    'data_config': data_config,
    'model': model_config,
    'lr': .001,
    'batch_size': edge_loader_train.batch_size,
    'epochs': 3
}


criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=experiment_config['lr'])
trainer_logic = GNNTrainingLogic(model, optimizer, criterion)
trainer_logic.train_step(None, batch)
None # don't push to stdout


In [21]:
from jazz_graph.training.logging import save_checkpoint_handler


experiment_logger = ExperimentLogger(root='../experiments', run_name=f'gnn_link_prediction_{os.path.basename(models_dir)}', config=experiment_config)

trainer = Engine(trainer_logic.train_step)
dev_evaluator = Engine(trainer_logic.eval_step)

metrics = {
    'accuracy': Accuracy(output_transform=binary_output_transform),
    'recall': Recall(output_transform=binary_output_transform),
    'precision': Precision(output_transform=binary_output_transform),
    'loss': Loss(criterion, output_transform=lambda out: (out['y_pred'], out['y_true']))
}

for name, metric in metrics.items():
    metric.attach(trainer, name)
    metric.attach(dev_evaluator, name)

progress_bar = ProgressBar()
progress_bar.attach(trainer)

trainer.add_event_handler(Events.EPOCH_COMPLETED, console_logging, 'Training', trainer)
trainer.add_event_handler(Events.EPOCH_COMPLETED, run_evaluator_handler, dev_evaluator, edge_loader_train)
trainer.add_event_handler(Events.EPOCH_COMPLETED, log_experiment_handler, experiment_logger, 'train', trainer)
dev_evaluator.add_event_handler(Events.EPOCH_COMPLETED, log_experiment_handler, experiment_logger, 'dev', trainer)
dev_evaluator.add_event_handler(Events.EPOCH_COMPLETED, console_logging, 'Valiation', trainer)
trainer.add_event_handler(Events.COMPLETED, save_checkpoint_handler, experiment_logger, model)
trainer.add_event_handler(Events.COMPLETED, save_embeddings_handler, experiment_logger, model)


In [22]:
trainer.run(edge_loader_train, max_epochs=15)

Epoch [1/15]: [1/30]   3%|▎          [00:00<?]

Training - Epoch[001]
  Avg. accuracy: 0.657;   Avg. recall: 0.340;   Avg. precision: 0.479;   Avg. loss: 0.802; 
Valiation - Epoch[001]
  Avg. accuracy: 0.715;   Avg. recall: 0.408;   Avg. precision: 0.609;   Avg. loss: 0.513; 


Training - Epoch[002]
  Avg. accuracy: 0.738;   Avg. recall: 0.479;   Avg. precision: 0.644;   Avg. loss: 0.483; 
Valiation - Epoch[002]
  Avg. accuracy: 0.784;   Avg. recall: 0.665;   Avg. precision: 0.679;   Avg. loss: 0.442; 


Training - Epoch[003]
  Avg. accuracy: 0.791;   Avg. recall: 0.639;   Avg. precision: 0.707;   Avg. loss: 0.419; 
Valiation - Epoch[003]
  Avg. accuracy: 0.828;   Avg. recall: 0.734;   Avg. precision: 0.745;   Avg. loss: 0.373; 


Training - Epoch[004]
  Avg. accuracy: 0.820;   Avg. recall: 0.708;   Avg. precision: 0.741;   Avg. loss: 0.374; 
Valiation - Epoch[004]
  Avg. accuracy: 0.858;   Avg. recall: 0.801;   Avg. precision: 0.780;   Avg. loss: 0.324; 


...

Training - Epoch[012]
  Avg. accuracy: 0.947;   Avg. recall: 0.941;   Avg. precision: 0.903;   Avg. loss: 0.142; 
Valiation - Epoch[012]
  Avg. accuracy: 0.956;   Avg. recall: 0.972;   Avg. precision: 0.903;   Avg. loss: 0.129; 


Training - Epoch[013]
  Avg. accuracy: 0.946;   Avg. recall: 0.934;   Avg. precision: 0.906;   Avg. loss: 0.144; 
Valiation - Epoch[013]
  Avg. accuracy: 0.956;   Avg. recall: 0.980;   Avg. precision: 0.898;   Avg. loss: 0.127; 


Training - Epoch[014]
  Avg. accuracy: 0.954;   Avg. recall: 0.948;   Avg. precision: 0.916;   Avg. loss: 0.132; 
Valiation - Epoch[014]
  Avg. accuracy: 0.964;   Avg. recall: 0.965;   Avg. precision: 0.930;   Avg. loss: 0.109; 


Training - Epoch[015]
  Avg. accuracy: 0.957;   Avg. recall: 0.955;   Avg. precision: 0.918;   Avg. loss: 0.121; 
Valiation - Epoch[015]
  Avg. accuracy: 0.968;   Avg. recall: 0.972;   Avg. precision: 0.934;   Avg. loss: 0.100; 
Saved embeddings to ../experiments/2026-03-03_12-44-44_gnn_link_prediction_graph_parquet_proto


State:
	iteration: 450
	epoch: 15
	epoch_length: 30
	max_epochs: 15
	output: <class 'dict'>
	batch: <class 'torch_geometric.data.hetero_data.HeteroData'>
	metrics: <class 'dict'>
	dataloader: <class 'torch_geometric.loader.link_neighbor_loader.LinkNeighborLoader'>
	seed: <class 'NoneType'>
	times: <class 'dict'>

In [ ]:
# Are val supervision edges present in the message passing graph?
val_supervise = set(map(tuple, dev_data['artist', 'performs', 'performance'].edge_label_index.t().tolist()))
val_msg_passing = set(map(tuple, dev_data['artist', 'performs', 'performance'].edge_index.t().tolist()))

leakage = val_supervise & val_msg_passing
print(f"Supervision edges also in message passing: {len(leakage)} / {len(val_supervise)}")

# Check reverse edges
val_reverse = set(map(tuple, dev_data['performance', 'rev_performs', 'artist'].edge_index.t().tolist()))
val_supervise_reversed = {(b, a) for a, b in val_supervise}
reverse_leakage = val_supervise_reversed & val_reverse
print(f"Supervision edges leaked via reverse edges: {len(reverse_leakage)} / {len(val_supervise)}")